# CineMatch — Exploratory Data Analysis & Data Quality Audit

**Author**: Senior ML Engineering & Data Science Team  
**Dataset**: GroupLens MovieLens Latest-Small  
**Status**: Fully Executed Real Data Pipeline

## 1. Environment & Data Ingestion

In [1]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ratings = pd.read_csv('data/raw/ml-latest-small/ratings.csv')
movies = pd.read_csv('data/raw/ml-latest-small/movies.csv')
tags = pd.read_csv('data/raw/ml-latest-small/tags.csv')
links = pd.read_csv('data/raw/ml-latest-small/links.csv')

print(f"Ratings shape: {ratings.shape}")
print(f"Movies shape:  {movies.shape}")
print(f"Tags shape:    {tags.shape}")
print(f"Links shape:   {links.shape}")

Ratings shape: (100836, 4)
Movies shape:  (9742, 3)
Tags shape:    (3683, 4)
Links shape:   (9742, 3)


## 2. Integrity, Nulls, and Duplicate Checks

In [2]:
print("Ratings nulls:", ratings.isnull().sum().to_dict())
print("Movies nulls: ", movies.isnull().sum().to_dict())
print("Tags nulls:   ", tags.isnull().sum().to_dict())
print("Links nulls:  ", links.isnull().sum().to_dict())

dup_ratings = ratings.duplicated(subset=['userId', 'movieId']).sum()
dup_movies = movies.duplicated(subset=['movieId']).sum()
print(f"Duplicate user-movie pairs: {dup_ratings}")
print(f"Duplicate movieId records:  {dup_movies}")

Ratings nulls: {'userId': 0, 'movieId': 0, 'rating': 0, 'timestamp': 0}
Movies nulls:  {'movieId': 0, 'title': 0, 'genres': 0}
Tags nulls:    {'userId': 0, 'movieId': 0, 'tag': 0, 'timestamp': 0}
Links nulls:   {'movieId': 0, 'imdbId': 0, 'tmdbId': 8}
Duplicate user-movie pairs: 0
Duplicate movieId records:  0


## 3. Matrix Sparsity Analysis

In [3]:
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)
total_possible = n_users * n_movies
sparsity = (1.0 - (n_ratings / total_possible)) * 100.0

print(f"Unique Users:        {n_users:,}")
print(f"Unique Rated Movies: {n_movies:,}")
print(f"Total Interactions:  {n_ratings:,}")
print(f"Matrix Sparsity:     {sparsity:.4f}%")

Unique Users:        610
Unique Rated Movies: 9,724
Total Interactions:  100,836
Matrix Sparsity:     98.3000%


## 4. Rating Distribution Statistics

In [4]:
rating_counts = ratings['rating'].value_counts().sort_index()
print("Rating Value Distribution:")
print(rating_counts)
print(f"Mean Rating:   {ratings['rating'].mean():.3f}")
print(f"Median Rating: {ratings['rating'].median():.3f}")
print(f"Std Dev:       {ratings['rating'].std():.3f}")

Rating Value Distribution:
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64
Mean Rating:   3.502
Median Rating: 3.500
Std Dev:       1.043


## 5. Activity & Popularity Long Tails

In [5]:
user_act = ratings.groupby('userId').size()
print("User Activity Percentiles:")
print(user_act.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

movie_pop = ratings.groupby('movieId').size()
print("\nMovie Popularity Percentiles:")
print(movie_pop.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

User Activity Percentiles:
count     610.000000
mean      165.304918
std       269.480584
min        20.000000
10%        24.000000
25%        35.000000
50%        70.500000
75%       168.000000
90%       400.300000
95%       610.750000
99%      1256.220000
max      2698.000000
dtype: float64

Movie Popularity Percentiles:
count    9724.000000
mean       10.369807
std        22.401005
min         1.000000
10%         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
90%        27.000000
95%        47.000000
99%       114.540000
max       329.000000
dtype: float64


## 6. Genre Diversity & Categorical Profiles

In [6]:
genre_counts = movies['genres'].str.split('|').explode().value_counts()
print("Genre Frequency across 9,742 Catalog Movies:")
print(genre_counts)

Genre Frequency across 9,742 Catalog Movies:
genres
Drama                 4361
Comedy                3756
Thriller              1894
Action                1828
Romance               1596
Adventure             1263
Crime                 1199
Sci-Fi                 980
Horror                 978
Fantasy                779
Children               664
Animation              611
Mystery                573
Documentary            440
War                    382
Musical                334
Western                167
IMAX                   158
Film-Noir               87
(no genres listed)      34
Name: count, dtype: int64


## 7. Temporal Range and Timestamp Validation

In [7]:
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
print(f"First rating timestamp: {ratings['datetime'].min()}")
print(f"Last rating timestamp:  {ratings['datetime'].max()}")
print(f"Temporal span: {(ratings['datetime'].max() - ratings['datetime'].min()).days} days (~22.5 years)")

First rating timestamp: 1996-03-29 18:36:55
Last rating timestamp:  2018-09-24 14:27:30
Temporal span: 8213 days (~22.5 years)
